# S1 — CMIP6 SSP global download and preprocessing / CMIP6 SSP全球数据下载与预处理

This notebook downloads and preprocesses CMIP6 SSP scenario data for all selected model families. It reads the download manifest produced by the SSP S0.1 notebook, queries ESGF for file-level records, and downloads missing files with checksum verification.

**Key differences from the historical S1:**
- **activity_id**: `ScenarioMIP` (not `CMIP`)
- **experiment_id**: one of `ssp126`, `ssp245`, `ssp585` (set via `CURRENT_SCENARIO`)
- **Time period**: 2015–2100 (not 1985–2014)
- **Fixed fields**: reused from historical downloads — no re-download needed
- Run the notebook once per scenario (change `CURRENT_SCENARIO`)

**Inputs:** `S0.1_ssp_download_manifest.csv` and `S0.1_selected_models.csv` from the SSP S0.1 notebook.

**中文说明：** 本Notebook下载CMIP6 SSP情景数据。与历史S1的区别：activity_id为ScenarioMIP，experiment_id为ssp126/245/585（通过`CURRENT_SCENARIO`设置），时间段为2015–2100，固定场复用历史下载。每次运行一个情景——修改`CURRENT_SCENARIO`后重新运行即可。

## 1. Configuration / 配置

Read the SSP S0.1 download manifest and selected models. `CURRENT_SCENARIO` selects which SSP to process in this run — change it and re-run for each scenario. `BATCH_FAMILIES` subsets which families to process. Fixed fields (sftlf, areacella, sftgif) are reused from the historical download at `DATA_ROOT/{model}/historical/...`.

**中文说明：** 从SSP S0.1读取下载清单。`CURRENT_SCENARIO`选择当前处理的SSP情景——改完后重新运行即可处理下一个。固定场从历史下载路径复用，不重新下载。

In [6]:
from __future__ import annotations

import json
import sys
import time
import traceback
import warnings
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display


def locate_case_dir() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / 'cmip_utils.py').exists() and candidate.name == 'caseA':
            return candidate
        nested = candidate / 'case' / 'caseA'
        if (nested / 'cmip_utils.py').exists():
            return nested
    raise FileNotFoundError('Could not locate case/caseA/cmip_utils.py')


CASE_DIR = locate_case_dir()
if str(CASE_DIR) not in sys.path:
    sys.path.insert(0, str(CASE_DIR))

from cmip_utils import (
    discover_file_records,
    download_record,
    harmonize_horizontal_grid,
    human_size,
    monthly_flux_to_annual_total,
    monthly_state_to_annual_mean,
    normalize_fraction_to_percent,
    normalize_longitude,
    open_time_series,
    processing_signature,
    select_fixed_record,
    validate_local_netcdf,
)

# ── Paths ──
FUTURE_DIR = Path.cwd().resolve()
if FUTURE_DIR.name != 'case_future2609':
    for d in [FUTURE_DIR, *FUTURE_DIR.parents]:
        nested = d / 'case' / 'case_future2609'
        if nested.is_dir():
            FUTURE_DIR = nested
            break

S01_DIR = FUTURE_DIR / 'output' / 'S0.1'
DATA_ROOT = Path('/Volumes/mimi-T9/CMIP6')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# ── Scenario selection ──
# Change CURRENT_SCENARIO and re-run to process each SSP scenario.
SCENARIOS = ['ssp126', 'ssp245', 'ssp585']
CURRENT_SCENARIO = 'ssp585'  # ← change this: 'ssp126', 'ssp245', or 'ssp585'

# ── Experiment ──
ACTIVITY_ID = 'ScenarioMIP'
EXPERIMENT_ID = CURRENT_SCENARIO
START_YEAR = 2015
END_YEAR = 2100

# ── Batch control ──
BATCH_FAMILIES = None  # None = all families
RUN_ONLY_MODELS = None
DISCOVERY_MODEL_DELAY_SECONDS = 2
DOWNLOAD_MODEL_DELAY_SECONDS = 10
DOWNLOAD_WORKERS = 1

EXTRA_MODELS = [
    {
        'model': 'GISS-E3-G', 'family': 'GISS-E3-G',
        'member_id': 'r1i1p101f1', 'pool': 'core',
        'n_target_available': 18, 'n_target_missing': 0, 'missing': '',
    },
    {
        'model': 'ACCESS-ESM1-5', 'family': 'ACCESS-ESM1',
        'member_id': 'r1i1p1f1', 'pool': 'core',
        'n_target_available': 17, 'n_target_missing': 1, 'missing': 'tran/Lmon',
    },
    {
        'model': 'AWI-ESM-1-1-LR', 'family': 'AWI',
        'member_id': 'r1i1p1f1', 'pool': 'core',
        'n_target_available': 17, 'n_target_missing': 1, 'missing': 'evspsblsoi/Lmon',
    },
    {
        'model': 'INM-CM4-8', 'family': 'INM',
        'member_id': 'r1i1p1f1', 'pool': 'core',
        'n_target_available': 15, 'n_target_missing': 3, 'missing': 'mrsos/Lmon, snw/LImon, tran/Lmon',
    },
    {
        'model': 'MPI-ESM1-2-LR', 'family': 'MPI',
        'member_id': 'r1i1p1f1', 'pool': 'core',
        'n_target_available': 16, 'n_target_missing': 2, 'missing': 'evspsblsoi/Lmon, tran/Lmon',
    },
    {
        'model': 'MIROC-ES2L', 'family': 'MIROC-ES2',
        'member_id': 'r1i1p1f2', 'pool': 'core',
        'n_target_available': 18, 'n_target_missing': 0, 'missing': '',
    },
    {
        'model': 'GFDL-ESM4', 'family': 'GFDL',
        'member_id': 'r1i1p1f1', 'pool': 'core',
        'n_target_available': 18, 'n_target_missing': 0, 'missing': '',
    },
    {
        'model': 'CNRM-ESM2-1', 'family': 'CNRM',
        'member_id': 'r1i1p1f2', 'pool': 'core',
        'n_target_available': 18, 'n_target_missing': 0, 'missing': '',
    },
]
EXTRA_MODEL_NAMES = {row['model'] for row in EXTRA_MODELS}

# ── Processing flags ──
SPATIAL_EXTENT = 'global'
DOWNLOAD_IF_MISSING = True
PROCESSING_VERSION = 'S1-global-v5-18vars-ssp'

# ── Variable specs (same as historical) ──
TIME_VARIABLES = {
    'P':    {'variable_id': 'pr',      'table_id': 'Amon', 'processing': 'flux_total', 'category': 'core'},
    'ET':   {'variable_id': 'evspsbl', 'table_id': 'Amon', 'processing': 'flux_total', 'category': 'core'},
    'R':    {'variable_id': 'mrro',    'table_id': 'Lmon', 'processing': 'flux_total', 'category': 'core'},
    'tas':  {'variable_id': 'tas',  'table_id': 'Amon', 'processing': 'state_mean', 'category': 'context'},
    'rsds': {'variable_id': 'rsds', 'table_id': 'Amon', 'processing': 'state_mean', 'category': 'context'},
    'rsus': {'variable_id': 'rsus', 'table_id': 'Amon', 'processing': 'state_mean', 'category': 'context'},
    'rlds': {'variable_id': 'rlds', 'table_id': 'Amon', 'processing': 'state_mean', 'category': 'context'},
    'rlus': {'variable_id': 'rlus', 'table_id': 'Amon', 'processing': 'state_mean', 'category': 'context'},
    'hfls': {'variable_id': 'hfls', 'table_id': 'Amon', 'processing': 'state_mean', 'category': 'context'},
    'hfss': {'variable_id': 'hfss', 'table_id': 'Amon', 'processing': 'state_mean', 'category': 'context'},
    'mrso': {'variable_id': 'mrso', 'table_id': 'Lmon', 'processing': 'state_mean', 'category': 'context'},
    'lai':  {'variable_id': 'lai',  'table_id': 'Lmon', 'processing': 'state_mean', 'category': 'context'},
    'mrsos':       {'variable_id': 'mrsos',       'table_id': 'Lmon', 'processing': 'state_mean', 'category': 'context'},
    'mrros':       {'variable_id': 'mrros',       'table_id': 'Lmon', 'processing': 'flux_total', 'category': 'context'},
    'prsn':        {'variable_id': 'prsn',        'table_id': 'Amon', 'processing': 'flux_total', 'category': 'context'},
    'snw':         {'variable_id': 'snw',         'table_id': 'LImon', 'processing': 'state_mean', 'category': 'context'},
    'evspsblsoi':  {'variable_id': 'evspsblsoi',  'table_id': 'Lmon', 'processing': 'flux_total', 'category': 'context'},
    'tran':        {'variable_id': 'tran',        'table_id': 'Lmon', 'processing': 'flux_total', 'category': 'context'},
}

FIXED_FIELDS = {
    'cell_area':         {'variable_id': 'areacella', 'table_id': 'fx', 'required': True},
    'land_fraction':     {'variable_id': 'sftlf',     'table_id': 'fx', 'required': True},
    'land_ice_fraction': {'variable_id': 'sftgif',    'table_id': 'fx', 'required': False},
}

# ── Load SSP S0.1 manifests ──
ssp_manifest_all = pd.read_csv(S01_DIR / 'S0.1_ssp_download_manifest.csv')
selected_models = pd.read_csv(S01_DIR / 'S0.1_selected_models.csv')

manifest_all = ssp_manifest_all[ssp_manifest_all['scenario'] == CURRENT_SCENARIO].copy()

if BATCH_FAMILIES is not None:
    selected_models = selected_models[selected_models['family'].isin(BATCH_FAMILIES)].copy()
    manifest_all = manifest_all[manifest_all['family'].isin(BATCH_FAMILIES)].copy()

extra_df = pd.DataFrame(EXTRA_MODELS)
for col in selected_models.columns:
    if col not in extra_df.columns:
        extra_df[col] = pd.NA
extra_df = extra_df.reindex(columns=list(selected_models.columns) + [
    c for c in extra_df.columns if c not in selected_models.columns
])
selected_models = pd.concat([selected_models, extra_df], ignore_index=True)
selected_models = selected_models.drop_duplicates(subset=['model'], keep='last')

if RUN_ONLY_MODELS is not None:
    run_only = list(dict.fromkeys(RUN_ONLY_MODELS))
    unknown = sorted(set(run_only) - set(selected_models['model']))
    if unknown:
        raise ValueError(f'RUN_ONLY_MODELS contains unknown models: {unknown}')
    selected_models = (
        selected_models.set_index('model', drop=False).loc[run_only].reset_index(drop=True)
    )
    manifest_all = manifest_all[manifest_all['model'].isin(run_only)].copy()

MODELS = selected_models['model'].tolist()
MODEL_INFO = selected_models.set_index('model')

print(f'Case directory: {CASE_DIR}')
print(f'Future S0.1 dir: {S01_DIR}')
print(f'Data root: {DATA_ROOT}')
print(f'=== CURRENT SCENARIO: {CURRENT_SCENARIO} ===')
print(f'Activity: {ACTIVITY_ID}, Period: {START_YEAR}–{END_YEAR}')
print(f'Batch families: {BATCH_FAMILIES}')
print(f'Run-only models: {RUN_ONLY_MODELS}')
print(f'Extra models: {sorted(EXTRA_MODEL_NAMES)}')
print(f'Total: {len(MODELS)} models')
print(f'Models: {MODELS}')
print(f'Manifest records for {CURRENT_SCENARIO}: {len(manifest_all)}')
display_cols = [c for c in [
    'model', 'family', 'member_id', 'pool',
    'n_target_available', 'n_target_missing', 'missing',
] if c in selected_models.columns]
display(selected_models[display_cols])

Case directory: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA
Future S0.1 dir: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/case_future2609/output/S0.1
Data root: /Volumes/mimi-T9/CMIP6
=== CURRENT SCENARIO: ssp585 ===
Activity: ScenarioMIP, Period: 2015–2100
Batch families: None
Run-only models: None
Extra models: ['ACCESS-ESM1-5', 'AWI-ESM-1-1-LR', 'CNRM-ESM2-1', 'GFDL-ESM4', 'GISS-E3-G', 'INM-CM4-8', 'MIROC-ES2L', 'MPI-ESM1-2-LR']
Total: 39 models
Models: ['CESM2', 'CNRM-CM6-1', 'CanESM5', 'E3SM-1-0', 'GFDL-CM4', 'IPSL-CM6A-LR', 'MIROC-ES2H', 'MRI-ESM2-0', 'SAM0-UNICON', 'TaiESM1', 'MIROC6', 'MPI-ESM-1-2-HAM', 'EC-Earth3-AerChem', 'FGOALS-f3-L', 'MCM-UA-1-0', 'BCC-CSM2-MR', 'GISS-E2-1-G', 'GISS-E2-2-G', 'UKESM1-0-LL', 'KACE-1-0-G', 'NorESM2-LM', 'CMCC-CM2-SR5', 'FGOALS-g3', 'ACCESS-CM2', 'ICON-ESM-LR', 'IPSL-CM5A2-INCA', 'CAMS-CSM1-0', 'CAS-ESM2-0', 'FIO-ESM-2-0', 'CIESM', 'KIOST-ESM', 'GISS-E3-G', 'ACCESS-ESM1-5', 'AWI-ESM-1-1-LR', 'INM-

,model,family,member_id,pool,n_target_available,n_target_missing,missing
0,CESM2,CESM2,r1i1p1f1,core,18,0,NaN
1,CNRM-CM6-1,CNRM,r1i1p1f2,core,18,0,NaN
2,CanESM5,CanESM,r1i1p1f1,core,18,0,NaN
3,E3SM-1-0,E3SM,r1i1p1f1,core,18,0,NaN
4,GFDL-CM4,GFDL,r1i1p1f1,core,18,0,NaN
5,IPSL-CM6A-LR,IPSL-CM6A,r1i1p1f1,core,18,0,NaN
6,MIROC-ES2H,MIROC-ES2,r1i1p4f2,core,18,0,NaN
7,MRI-ESM2-0,MRI,r1i2p1f1,core,18,0,NaN
8,SAM0-UNICON,SAM0,r1i1p1f1,core,18,0,NaN
9,TaiESM1,TaiESM,r1i1p1f1,core,18,0,NaN


## 2. Discover file-level records from ESGF / 查询ESGF文件级记录

For each model, query ESGF for the SSP scenario's file-level records (URLs, checksums, sizes). Fixed fields (areacella, sftlf, sftgif) are **not queried** — they are reused from the historical download at `DATA_ROOT/{model}/historical/{member}/{grid}/raw/`.

**中文说明：** 对每个模型查询ESGF的SSP文件级记录。固定场不重新查询——复用历史下载路径。若某个模型在该SSP情景下没有数据，则跳过。

In [7]:
import gc
from collections import Counter

def _merge_and_save(new_df: pd.DataFrame, path: Path, key_cols: list[str]) -> pd.DataFrame:
    """Merge new rows into an existing CSV, replacing rows with matching keys."""
    if path.exists():
        existing = pd.read_csv(path)
        existing = existing[~existing.set_index(key_cols).index.isin(new_df.set_index(key_cols).index)]
        merged = pd.concat([existing, new_df], ignore_index=True)
    else:
        merged = new_df
    merged.to_csv(path, index=False)
    return merged


def _find_historical_fixed(model, member_id, grid_label, variable_id):
    """Find a fixed field file from the historical download."""
    hist_raw = DATA_ROOT / model / 'historical' / member_id / grid_label / 'raw' / variable_id
    if hist_raw.is_dir():
        ncs = sorted(hist_raw.glob('*.nc'))
        if ncs:
            return ncs[0]
    for hist_member_dir in sorted((DATA_ROOT / model / 'historical').glob('*')):
        if not hist_member_dir.is_dir():
            continue
        for hist_grid_dir in sorted(hist_member_dir.glob('*')):
            raw_dir = hist_grid_dir / 'raw' / variable_id
            if raw_dir.is_dir():
                ncs = sorted(raw_dir.glob('*.nc'))
                if ncs:
                    return ncs[0]
    return None


def _probe_ssp_members(model, experiment_id, activity_id):
    """Query ESGF without member constraint to find available members for this SSP."""
    probe_records = discover_file_records(
        source_id=model,
        experiment_id=experiment_id,
        variable_id='pr',
        allowed_tables=['Amon'],
        requested_start_year=START_YEAR,
        requested_end_year=END_YEAR,
        fixed=False,
        member_id=None,
        activity_id=activity_id,
    )
    if not probe_records:
        probe_records = discover_file_records(
            source_id=model,
            experiment_id=experiment_id,
            variable_id='tas',
            allowed_tables=['Amon'],
            requested_start_year=START_YEAR,
            requested_end_year=END_YEAR,
            fixed=False,
            member_id=None,
            activity_id=activity_id,
        )
    member_counts = Counter(r.member_id for r in probe_records if r.member_id)
    return member_counts


def _pick_best_member(member_counts, preferred='r1i1p1f1'):
    """Pick the best available member, preferring the standard r1i1p1f1."""
    if not member_counts:
        return None
    if preferred in member_counts:
        return preferred
    def _sort_key(m):
        parts = m.replace('r', '').replace('i', ' ').replace('p', ' ').replace('f', ' ').split()
        try:
            nums = tuple(int(x) for x in parts)
        except ValueError:
            nums = (999, 999, 999, 999)
        return nums
    return min(member_counts, key=_sort_key)


def _discover_model_variables(model, member_id, model_time_vars, model_manifest,
                              allow_esgf_without_manifest, grid_label_hint=None):
    """Run ESGF discovery for all time variables of one model. Returns (records_by_alias, discovery_errors, grid_label)."""
    records_by_alias = {}
    discovery_errors = {}
    grid_label = grid_label_hint

    for alias, spec in model_time_vars.items():
        var_id = spec['variable_id']
        table_id = spec['table_id']
        var_rows = model_manifest[model_manifest['variable'] == var_id]
        if var_rows.empty and not allow_esgf_without_manifest:
            continue
        gl = str(var_rows.iloc[0]['grid_label']).split(',')[0].strip() if not var_rows.empty else grid_label
        if grid_label is None and gl:
            grid_label = gl

        try:
            records = discover_file_records(
                source_id=model,
                experiment_id=EXPERIMENT_ID,
                variable_id=var_id,
                allowed_tables=[table_id],
                requested_start_year=START_YEAR,
                requested_end_year=END_YEAR,
                fixed=False,
                member_id=member_id,
                activity_id=ACTIVITY_ID,
            )
            if gl:
                records = [
                    r for r in records
                    if r.member_id == member_id and r.grid_label == gl
                ]
            else:
                records = [r for r in records if r.member_id == member_id]
                if records and grid_label is None:
                    grid_label = records[0].grid_label
                    gl = grid_label
                    records = [r for r in records if r.grid_label == gl]
        except Exception as exc:
            records = []
            discovery_errors[alias] = str(exc)
        records_by_alias[alias] = records
        print(f'  {alias} ({var_id}/{table_id}): {len(records)} files on {gl}')

    return records_by_alias, discovery_errors, grid_label


RUNS = []
plan_rows = []
discovery_t0 = time.time()

for model in MODELS:
    info = MODEL_INFO.loc[model]
    member_id = info['member_id']
    pool = info['pool']
    model_manifest = manifest_all[manifest_all['model'] == model]
    available_vars = set(model_manifest['variable'].values)
    allow_esgf_without_manifest = (
        model in EXTRA_MODEL_NAMES or model_manifest.empty
    )

    print(f'\n{"="*60}')
    print(f'[{CURRENT_SCENARIO}] Resolving {model} (family={info["family"]}, pool={pool}, member={member_id})')

    model_time_vars = {}
    for alias, spec in TIME_VARIABLES.items():
        var_id = spec['variable_id']
        if var_id == 'evspsbl' and pool == 'P+R':
            continue
        if var_id not in available_vars and not allow_esgf_without_manifest:
            print(f'  {alias} ({var_id}): not in manifest')
            continue
        model_time_vars[alias] = spec

    records_by_alias, discovery_errors, grid_label = _discover_model_variables(
        model, member_id, model_time_vars, model_manifest, allow_esgf_without_manifest
    )

    # Member fallback: if all records empty, probe for available SSP members
    member_substituted = False
    original_member = member_id
    total_found = sum(len(recs) for recs in records_by_alias.values())
    if total_found == 0:
        print(f'  → 0 files with member={member_id}, probing ESGF for available members...')
        try:
            member_counts = _probe_ssp_members(model, EXPERIMENT_ID, ACTIVITY_ID)
            if member_counts:
                best = _pick_best_member(member_counts, preferred='r1i1p1f1')
                print(f'  → Available members: {dict(member_counts)}')
                print(f'  → Retrying with member={best} (was {member_id})')
                member_id = best
                member_substituted = True
                records_by_alias, discovery_errors, grid_label = _discover_model_variables(
                    model, member_id, model_time_vars, model_manifest,
                    allow_esgf_without_manifest=True, grid_label_hint=None
                )
            else:
                print(f'  → No ScenarioMIP data found for {model}/{CURRENT_SCENARIO}')
        except Exception as exc:
            print(f'  → Member probe failed: {exc}')

    if grid_label is None:
        grid_label = 'gn'

    # Fixed fields: reuse from historical download (no ESGF query)
    fixed_paths = {}
    fixed_missing_names = []
    for alias, spec in FIXED_FIELDS.items():
        var_id = spec['variable_id']
        path = _find_historical_fixed(model, member_id, grid_label, var_id)
        if path is None:
            path = _find_historical_fixed(model, original_member, grid_label, var_id)
        fixed_paths[alias] = path
        status_str = f'found at {path.name}' if path else 'MISSING from historical'
        print(f'  {alias} ({var_id}): {status_str}')
        if path is None and spec.get('required', False):
            fixed_missing_names.append(var_id)

    missing_required = list(fixed_missing_names)
    if 'P' not in records_by_alias or not records_by_alias['P']:
        missing_required.append('P')
    if 'R' not in records_by_alias or not records_by_alias['R']:
        missing_required.append('R')
    if pool == 'core' and ('ET' not in records_by_alias or not records_by_alias['ET']):
        missing_required.append('ET')

    if missing_required:
        status = 'missing_required'
        print(f'  *** SKIPPED: missing required fields: {missing_required}')
    else:
        status = 'ready'

    optional_missing = [
        alias for alias in model_time_vars
        if alias not in records_by_alias or not records_by_alias[alias]
    ]
    for alias in TIME_VARIABLES:
        if alias not in model_time_vars and alias not in optional_missing:
            optional_missing.append(alias)

    n_files = sum(len(recs) for recs in records_by_alias.values())
    total_bytes = sum(r.size for recs in records_by_alias.values() for r in recs)

    run_root = DATA_ROOT / model / EXPERIMENT_ID / member_id / (grid_label or 'gn')

    plan_row = {
        'scenario': CURRENT_SCENARIO,
        'model': model,
        'family': info['family'],
        'pool': pool,
        'member': member_id,
        'grid': grid_label,
        'n_time_vars': len(records_by_alias),
        'n_files': n_files,
        'download_size': human_size(total_bytes),
        'optional_missing': ', '.join(optional_missing),
        'fixed_missing': ', '.join(a for a, p in fixed_paths.items() if p is None),
        'status': status,
        'errors': '; '.join(f'{k}: {v}' for k, v in discovery_errors.items()),
    }
    if member_substituted:
        plan_row['member_note'] = f'fallback from {original_member}'
    plan_rows.append(plan_row)

    if status == 'ready':
        RUNS.append({
            'model': model,
            'family': info['family'],
            'pool': pool,
            'member_id': member_id,
            'grid_label': grid_label,
            'records_by_alias': records_by_alias,
            'fixed_paths': fixed_paths,
            'run_root': run_root,
            'model_time_vars': model_time_vars,
        })
    gc.collect()
    if DISCOVERY_MODEL_DELAY_SECONDS > 0:
        print(f'  discovery cooldown: {DISCOVERY_MODEL_DELAY_SECONDS}s')
        time.sleep(DISCOVERY_MODEL_DELAY_SECONDS)

discovery_elapsed = time.time() - discovery_t0
print(f'\nDiscovery complete in {discovery_elapsed:.0f}s ({discovery_elapsed/60:.1f} min)')

PLAN = pd.DataFrame(plan_rows)
print(PLAN.to_string(index=False))
if not PLAN.empty:
    PLAN = _merge_and_save(PLAN, DATA_ROOT / f'S1_run_plan_{CURRENT_SCENARIO}.csv', ['model'])
    print(f'{len(RUNS)} models ready for download out of {len(MODELS)} requested.')
    n_fallback = sum(1 for r in plan_rows if 'member_note' in r)
    if n_fallback:
        print(f'  ({n_fallback} used member fallback)')
else:
    print('No runnable models found.')
print('Discovery cell finished. Run the download cell next — do not Run All.')

del plan_rows
gc.collect()


[ssp585] Resolving CESM2 (family=CESM2, pool=core, member=r1i1p1f1)
  P (pr/Amon): 0 files on gn
  ET (evspsbl/Amon): 0 files on gn
  R (mrro/Lmon): 0 files on gn
  tas (tas/Amon): 0 files on gn
  rsds (rsds/Amon): 0 files on gn
  rsus (rsus/Amon): 0 files on gn
  rlds (rlds/Amon): 0 files on gn
  rlus (rlus/Amon): 0 files on gn
  hfls (hfls/Amon): 0 files on gn
  hfss (hfss/Amon): 0 files on gn
  mrso (mrso/Lmon): 0 files on gn
  lai (lai/Lmon): 0 files on gn
  mrsos (mrsos/Lmon): 0 files on gn
  mrros (mrros/Lmon): 0 files on gn
  prsn (prsn/Amon): 0 files on gn
  snw (snw/LImon): 0 files on gn
  evspsblsoi (evspsblsoi/Lmon): 0 files on gn
  tran (tran/Lmon): 0 files on gn
  → 0 files with member=r1i1p1f1, probing ESGF for available members...
  → Available members: {'r10i1p1f1': 2, 'r11i1p1f1': 2, 'r4i1p1f1': 2}
  → Retrying with member=r4i1p1f1 (was r1i1p1f1)
  P (pr/Amon): 2 files on gn
  ET (evspsbl/Amon): 2 files on gn
  R (mrro/Lmon): 2 files on gn
  tas (tas/Amon): 2 files on

0

## 3. Download files / 下载文件

Download SSP time-variable files only — fixed fields are reused from historical. A cached file is reused after size/checksum/NetCDF validation. `S1_download_status_{scenario}.csv` tracks file-level progress.

**中文说明：** 仅下载SSP时变变量文件——固定场复用历史下载。已有文件经验证后复用。下载状态持续写入CSV，kernel中断后可查看进度。

In [8]:
import gc
from concurrent.futures import ThreadPoolExecutor, as_completed


DOWNLOAD_STATUS_PATH = DATA_ROOT / f'S1_download_status_{CURRENT_SCENARIO}.csv'

REQUIRED_TIME = {'P', 'R'}


def obtain_local_file(record, target_dir: Path) -> Path:
    target = target_dir / record.title
    if DOWNLOAD_IF_MISSING:
        return download_record(record, target_dir)
    validation = validate_local_netcdf(target, record)
    if not validation.ok:
        raise FileNotFoundError(
            f'Download disabled and cached file is not valid: {target}; '
            + '; '.join(validation.messages)
        )
    record.local_path = str(target)
    record.status = 'existing-verified'
    record.warnings.extend(validation.warnings)
    return target


def _download_one(item):
    kind, alias, record, target_dir = item
    path = obtain_local_file(record, target_dir)
    return kind, alias, record.title, path


def _is_required(alias: str, pool: str) -> bool:
    if alias in REQUIRED_TIME:
        return True
    if alias == 'ET' and pool == 'core':
        return True
    return False


download_t0 = time.time()
FAILED_DOWNLOADS = []

for i, run in enumerate(RUNS):
    model = run['model']
    pool = run['pool']
    raw_root = run['run_root'] / 'raw'
    raw_root.mkdir(parents=True, exist_ok=True)
    local_paths = {}

    print(f'\n[{i+1}/{len(RUNS)}] Downloading {model} [{CURRENT_SCENARIO}] → {run["run_root"]}')
    model_t0 = time.time()

    try:
        jobs = []
        for alias, records in run['records_by_alias'].items():
            var_id = run['model_time_vars'][alias]['variable_id']
            for record in records:
                jobs.append(('time', alias, record, raw_root / var_id))

        results = {}
        pending = []
        for kind, alias, record, target_dir in jobs:
            target = target_dir / record.title
            if target.exists():
                path = obtain_local_file(record, target_dir)
                results[(kind, alias, record.title)] = path
            else:
                pending.append((kind, alias, record, target_dir))

        required_errors = []
        optional_errors = []
        if pending:
            with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool_ex:
                futures = {pool_ex.submit(_download_one, job): job for job in pending}
                for future in as_completed(futures):
                    kind, alias, record, _target_dir = futures[future]
                    try:
                        _kind, _alias, title, path = future.result()
                        results[(kind, alias, title)] = path
                    except Exception as exc:
                        short = f'{alias}/{record.variable_id}: {exc}'
                        if _is_required(alias, pool):
                            required_errors.append(short)
                        else:
                            optional_errors.append(short)
                            print(f'  skip optional {short}')
            del futures

        if required_errors:
            raise RuntimeError(' ; '.join(required_errors[:8]))

        for alias, records in run['records_by_alias'].items():
            alias_paths = [
                results[('time', alias, record.title)]
                for record in records
                if ('time', alias, record.title) in results
            ]
            if not alias_paths:
                if _is_required(alias, pool):
                    raise RuntimeError(f'required {alias} has no files')
                print(f'  {alias}: skipped (download failed)')
                continue
            local_paths[alias] = alias_paths
            print(
                f'  {alias}: {len(alias_paths)}/{len(records)} files '
                f'({records[0].status if records else "?"})'
            )

        # Fixed fields: use historical paths directly
        for alias, path in run['fixed_paths'].items():
            local_paths[alias] = path
            if path:
                print(f'  {alias}: reused from historical ({path.name})')
            else:
                print(f'  {alias}: not available')

        run['local_paths'] = local_paths

        # Audit log
        audit_rows = []
        for alias, records in run['records_by_alias'].items():
            for record in records:
                audit_rows.append({
                    'alias': alias, 'variable_id': record.variable_id,
                    'table_id': record.table_id, 'title': record.title,
                    'status': record.status, 'local_path': record.local_path,
                    'checksum_available': bool(record.checksum and record.checksum_type),
                    'warnings': ' | '.join(record.warnings),
                })
        qc_dir = run['run_root'] / 'qc'
        qc_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(audit_rows).to_csv(qc_dir / 'raw_file_audit.csv', index=False)

        print(f'  Done in {time.time() - model_t0:.0f}s')
        if optional_errors:
            print(f'  optional skipped: {len(optional_errors)}')

    except Exception as exc:
        print(f'  *** FAILED after {time.time() - model_t0:.0f}s: {exc}')
        FAILED_DOWNLOADS.append((model, str(exc)[:300]))
        run['local_paths'] = None

    del results, pending, jobs
    gc.collect()
    if DOWNLOAD_MODEL_DELAY_SECONDS > 0 and i < len(RUNS) - 1:
        print(f'  download cooldown: {DOWNLOAD_MODEL_DELAY_SECONDS}s')
        time.sleep(DOWNLOAD_MODEL_DELAY_SECONDS)

RUNS = [r for r in RUNS if r.get('local_paths') is not None]

print(f'\nDownloads complete in {time.time() - download_t0:.0f}s')
print(f'{len(RUNS)} succeeded, {len(FAILED_DOWNLOADS)} failed')
for model, err in FAILED_DOWNLOADS:
    print(f'  FAILED: {model}: {err}')


[1/23] Downloading CESM2 [ssp585] → /Volumes/mimi-T9/CMIP6/CESM2/ssp585/r4i1p1f1/gn


/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/xarray/conventions.py:286: SerializationWarning: variable 'pr' has multiple fill values {1e+20, 1e+20} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


    pr: 99.3 MiB from esgf-node.ornl.gov in 104s (1.00 MB/s) downloaded-verified


/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/xarray/conventions.py:286: SerializationWarning: variable 'pr' has multiple fill values {1e+20, 1e+20} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


    pr: 71.5 MiB from esgf-node.ornl.gov in 145s (0.52 MB/s) downloaded-verified


...

    evspsblsoi: 32.3 MiB from esgf-node.ornl.gov in 6s (5.87 MB/s) downloaded-verified


/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/xarray/conventions.py:286: SerializationWarning: variable 'tran' has multiple fill values {1e+20, 1e+20} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


    tran: 33.5 MiB from esgf-node.ornl.gov in 8s (4.66 MB/s) downloaded-verified


/Users/mimi/miniconda3/envs/pip39/lib/python3.9/site-packages/xarray/conventions.py:286: SerializationWarning: variable 'tran' has multiple fill values {1e+20, 1e+20} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


    tran: 24.4 MiB from esgf-node.ornl.gov in 5s (5.52 MB/s) downloaded-verified
  P: 2/2 files (downloaded-verified)
  ET: 2/2 files (downloaded-verified)
  R: 2/2 files (downloaded-verified)
  tas: 2/2 files (downloaded-verified)
  rsds: 2/2 files (downloaded-verified)
  rsus: 2/2 files (downloaded-verified)
  rlds: 2/2 files (downloaded-verified)
  rlus: 2/2 files (downloaded-verified)
  hfls: 2/2 files (downloaded-verified)
  hfss: 2/2 files (downloaded-verified)
  mrso: 2/2 files (downloaded-verified)
  lai: 2/2 files (downloaded-verified)
  mrsos: 2/2 files (downloaded-verified)
  mrros: 2/2 files (downloaded-verified)
  prsn: 2/2 files (downloaded-verified)
  snw: 2/2 files (downloaded-verified)
  evspsblsoi: 2/2 files (downloaded-verified)
  tran: 2/2 files (downloaded-verified)
  cell_area: reused from historical (areacella_fx_CESM2_historical_r1i1p1f1_gn.nc)
  land_fraction: reused from historical (sftlf_fx_CESM2_historical_r1i1p1f1_gn.nc)
  land_ice_fraction: reused from his

## 4. Build annual and climatological products / 构建年度与气候态产品

*(Processing is commented out — enable when ready.)*

Monthly water fluxes (pr, evspsbl, mrro) are integrated to annual totals (mm yr⁻¹). Context variables (tas, radiation, heat fluxes, mrso, lai) are averaged to annual means. Longitude labels are normalized to 0–360. Processed products are rebuilt only when their source/configuration signature changes.

**中文说明：** （处理代码已注释——准备好后启用。）月尺度水通量按calendar积分为年总量；context变量按月天数加权计算年平均。统一经度坐标后检查所有变量网格一致性。合并areacella、sftlf和sftgif，计算陆地面积。P+R模型没有ET，因此不计算WB残差。已有processed产品在签名一致时直接复用。

In [9]:
# import gc


# def _atomic_netcdf(dataset: xr.Dataset, path: Path) -> None:
#     path.parent.mkdir(parents=True, exist_ok=True)
#     temporary = path.with_suffix(path.suffix + '.tmp')
#     dataset.to_netcdf(temporary)
#     temporary.replace(path)


# def _atomic_parquet(frame: pd.DataFrame, path: Path) -> None:
#     path.parent.mkdir(parents=True, exist_ok=True)
#     temporary = path.with_suffix(path.suffix + '.tmp')
#     frame.to_parquet(temporary, index=False)
#     temporary.replace(path)


# def _load_fixed(path: Path, variable_id: str) -> xr.DataArray:
#     with xr.open_dataset(path) as dataset:
#         if variable_id not in dataset:
#             raise KeyError(f'{variable_id!r} absent from {path}')
#         return normalize_longitude(dataset[variable_id].load())


# def _signature_payload(run: dict) -> dict:
#     fixed_sig = {}
#     for alias, path in run['fixed_paths'].items():
#         if path is None:
#             fixed_sig[alias] = None
#         else:
#             fixed_sig[alias] = {'path': str(path), 'size': path.stat().st_size}
#     return {
#         'processing_version': PROCESSING_VERSION,
#         'model': run['model'],
#         'experiment': EXPERIMENT_ID,
#         'member': run['member_id'],
#         'grid': run['grid_label'],
#         'period': [START_YEAR, END_YEAR],
#         'time_records': {
#             alias: [
#                 {key: getattr(record, key) for key in ('title', 'version', 'size', 'checksum', 'checksum_type')}
#                 for record in records
#             ]
#             for alias, records in run['records_by_alias'].items()
#         },
#         'fixed_paths': fixed_sig,
#     }


# def process_run(run: dict) -> tuple[int, str]:
#     """Process one model: build annual + climatology products, save to disk.

#     Returns (n_time_vars, status).  Datasets are written to disk and then
#     released so the batch loop never accumulates them in memory.
#     """
#     processed_dir = run['run_root'] / 'processed'
#     processed_dir.mkdir(parents=True, exist_ok=True)
#     annual_path = processed_dir / f'global_annual_{START_YEAR}_{END_YEAR}.nc'
#     climatology_path = processed_dir / f'global_climatology_{START_YEAR}_{END_YEAR}.nc'
#     parquet_path = processed_dir / f'global_climatology_{START_YEAR}_{END_YEAR}.parquet'
#     signature_path = processed_dir / f'global_processing_{START_YEAR}_{END_YEAR}.json'
#     signature = processing_signature(_signature_payload(run))

#     cache_valid = False
#     if all(p.exists() for p in [annual_path, climatology_path, parquet_path, signature_path]):
#         try:
#             cache_valid = json.loads(signature_path.read_text())['signature'] == signature
#         except Exception:
#             cache_valid = False
#     if cache_valid:
#         fixed_and_derived = {'cell_area', 'land_fraction', 'land_ice_fraction',
#                              'land_area', 'land_ice_area', 'non_ice_land_area', 'WB'}
#         with xr.open_dataset(annual_path) as ds:
#             n_vars = len([v for v in ds.data_vars if v not in fixed_and_derived])
#         return n_vars, 'existing-processed'

#     model_time_vars = run['model_time_vars']

#     annual_arrays = {}
#     for alias, spec in model_time_vars.items():
#         paths = run['local_paths'].get(alias)
#         if not paths:
#             continue
#         monthly = open_time_series(
#             paths, spec['variable_id'], start_year=START_YEAR, end_year=END_YEAR
#         )
#         monthly = normalize_longitude(monthly)
#         if spec['processing'] == 'flux_total':
#             annual = monthly_flux_to_annual_total(monthly)
#         elif spec['processing'] == 'state_mean':
#             annual = monthly_state_to_annual_mean(monthly)
#         else:
#             raise ValueError(f"Unknown processing rule: {spec['processing']}")
#         annual.name = alias
#         annual_arrays[alias] = annual
#         del monthly

#     ref_name = 'P'
#     annual_arrays = harmonize_horizontal_grid(annual_arrays, reference_name=ref_name)
#     expected_years = np.arange(START_YEAR, END_YEAR + 1)
#     for alias, annual in annual_arrays.items():
#         if not np.array_equal(annual['year'].values, expected_years):
#             raise ValueError(f'{alias} does not contain every requested year')
#     annual_ds = xr.Dataset(annual_arrays)

#     fixed_arrays = {}
#     for alias, spec in FIXED_FIELDS.items():
#         path = run['local_paths'].get(alias)
#         if path is None:
#             continue
#         fixed_arrays[alias] = _load_fixed(path, spec['variable_id'])

#     if 'cell_area' not in fixed_arrays or 'land_fraction' not in fixed_arrays:
#         raise ValueError('cell_area and land_fraction are required')
#     fixed_arrays['land_fraction'] = normalize_fraction_to_percent(
#         fixed_arrays['land_fraction'], name='sftlf'
#     )
#     if 'land_ice_fraction' in fixed_arrays:
#         fixed_arrays['land_ice_fraction'] = normalize_fraction_to_percent(
#             fixed_arrays['land_ice_fraction'], name='sftgif'
#         )

#     ref_and_fixed = {ref_name: annual_arrays[ref_name], **fixed_arrays}
#     harmonized = harmonize_horizontal_grid(ref_and_fixed, reference_name=ref_name)
#     fixed_arrays = {a: harmonized[a] for a in fixed_arrays}
#     del ref_and_fixed, harmonized

#     cell_area = fixed_arrays['cell_area'].astype('float64')
#     area_units = str(cell_area.attrs.get('units', '')).lower().replace(' ', '').replace('**', '^')
#     if area_units not in {'m2', 'm^2'}:
#         raise ValueError(f"Expected areacella in m2, found units={cell_area.attrs.get('units')!r}")
#     if float(cell_area.min(skipna=True)) <= 0:
#         raise ValueError('areacella must be positive')

#     land_fraction = fixed_arrays['land_fraction']
#     land_area = cell_area * land_fraction / 100.0
#     land_area.attrs = {'long_name': 'land-covered grid-cell area', 'units': 'm2'}

#     if 'land_ice_fraction' in fixed_arrays:
#         land_ice_fraction = fixed_arrays['land_ice_fraction']
#         overlap = np.isfinite(land_fraction) & np.isfinite(land_ice_fraction)
#         if bool(((land_ice_fraction > land_fraction + 1e-5) & overlap).any()):
#             raise ValueError('sftgif exceeds sftlf on one or more aligned cells')
#         land_ice_area = cell_area * land_ice_fraction / 100.0
#         non_ice_land_area = cell_area * (land_fraction - land_ice_fraction).clip(0.0, 100.0) / 100.0
#         land_ice_area.attrs = {'long_name': 'land-ice-covered grid-cell area', 'units': 'm2'}
#         non_ice_land_area.attrs = {'long_name': 'non-ice land grid-cell area', 'units': 'm2'}
#         land_ice_status = 'available'
#     else:
#         land_ice_fraction = xr.full_like(land_fraction, np.nan)
#         land_ice_fraction.attrs = {'long_name': 'land ice area percentage', 'units': '%', 'availability': 'unavailable'}
#         land_ice_area = xr.full_like(cell_area, np.nan)
#         non_ice_land_area = xr.full_like(cell_area, np.nan)
#         land_ice_status = 'unavailable'

#     annual_ds['cell_area'] = cell_area
#     annual_ds['land_fraction'] = land_fraction
#     annual_ds['land_ice_fraction'] = land_ice_fraction
#     annual_ds['land_area'] = land_area
#     annual_ds['land_ice_area'] = land_ice_area
#     annual_ds['non_ice_land_area'] = non_ice_land_area

#     if {'P', 'ET', 'R'} <= set(annual_ds.data_vars):
#         annual_ds['WB'] = annual_ds['P'] - annual_ds['ET'] - annual_ds['R']
#         annual_ds['WB'].attrs = {'long_name': 'apparent water-balance residual P-ET-R', 'units': 'mm yr-1'}

#     n_time_vars = len(annual_arrays)

#     time_var_names = list(annual_arrays.keys())
#     climatology_ds = annual_ds[time_var_names].mean('year', keep_attrs=True)
#     for name in ('cell_area', 'land_fraction', 'land_ice_fraction', 'land_area', 'land_ice_area', 'non_ice_land_area'):
#         climatology_ds[name] = annual_ds[name]
#     if 'WB' in annual_ds:
#         climatology_ds['WB'] = climatology_ds['P'] - climatology_ds['ET'] - climatology_ds['R']
#         climatology_ds['WB'].attrs = {'long_name': 'apparent water-balance residual P-ET-R', 'units': 'mm yr-1'}

#     common_attrs = {
#         'source_id': run['model'],
#         'experiment_id': EXPERIMENT_ID,
#         'member_id': run['member_id'],
#         'grid_label': run['grid_label'],
#         'spatial_extent': SPATIAL_EXTENT,
#         'analysis_period': f'{START_YEAR}-{END_YEAR}',
#         'land_ice_status': land_ice_status,
#         'processing_version': PROCESSING_VERSION,
#         'pool': run['pool'],
#         'evspsbl_sign_policy': 'original CMIP6 sign preserved',
#     }
#     annual_ds.attrs.update(common_attrs)
#     climatology_ds.attrs.update(common_attrs)

#     _atomic_netcdf(annual_ds, annual_path)
#     _atomic_netcdf(climatology_ds, climatology_path)

#     table = climatology_ds.to_dataframe().reset_index()
#     table.insert(0, 'model', run['model'])
#     table.insert(1, 'experiment', EXPERIMENT_ID)
#     table.insert(2, 'member', run['member_id'])
#     table.insert(3, 'grid', run['grid_label'])
#     table.insert(4, 'period_start', START_YEAR)
#     table.insert(5, 'period_end', END_YEAR)
#     _atomic_parquet(table, parquet_path)

#     signature_path.write_text(
#         json.dumps({'signature': signature, 'payload': _signature_payload(run)}, indent=2),
#         encoding='utf-8',
#     )

#     del annual_ds, climatology_ds, annual_arrays, fixed_arrays, table
#     gc.collect()

#     return n_time_vars, 'processed-now'


# process_t0 = time.time()
# PROCESSED = []
# FAILED_PROCESSING = []

# for i, run in enumerate(RUNS):
#     model = run['model']
#     print(f'[{i+1}/{len(RUNS)}] Processing {model} [{CURRENT_SCENARIO}]...')
#     t0 = time.time()
#     try:
#         n_vars, status = process_run(run)
#         run['processed_status'] = status
#         PROCESSED.append(run)
#         elapsed = time.time() - t0
#         print(f'  {model}: {status} ({n_vars} vars, {elapsed:.0f}s)')
#     except Exception as exc:
#         elapsed = time.time() - t0
#         print(f'  *** FAILED after {elapsed:.0f}s: {exc}')
#         traceback.print_exc()
#         FAILED_PROCESSING.append((model, str(exc)))

# total_process = time.time() - process_t0
# print(f'\nProcessing complete in {total_process:.0f}s ({total_process/60:.1f} min)')
# print(f'{len(PROCESSED)} succeeded, {len(FAILED_PROCESSING)} failed')
# if FAILED_PROCESSING:
#     for model, err in FAILED_PROCESSING:
#         print(f'  FAILED: {model}: {err}')

## 5. QC summaries / 质量检查汇总

*(QC is commented out — enable after processing.)*

QC is descriptive only. It reports completeness, value ranges, and negative ET frequencies for the core water-cycle variables. Context variables get basic range checks. Maps are drawn for the core variables only (P, ET, R, WB).

Both `global_climatology_qc_summary.csv` and `S1_processed_summary.csv` are **merged** with any existing file from previous runs, so incremental batches accumulate rather than overwrite.

**中文说明：** （QC代码已注释——处理完成后启用。）QC只检查数据完整性和数值合理性。Core变量（P/ET/R/WB）生成全球地图和详细统计；context变量只做基本范围检查。两个汇总表都是增量合并——多次分批运行不会丢失之前的记录。

In [10]:
# FIXED_VAR_NAMES = {'cell_area', 'land_fraction', 'land_ice_fraction', 'land_area', 'land_ice_area', 'non_ice_land_area'}


# def robust_limits(data: xr.DataArray, *, symmetric: bool = False) -> tuple[float, float]:
#     values = np.asarray(data.values)
#     values = values[np.isfinite(values)]
#     if values.size == 0:
#         return (-1.0, 1.0) if symmetric else (0.0, 1.0)
#     if symmetric:
#         limit = float(np.quantile(np.abs(values), 0.99))
#         if not np.isfinite(limit) or limit == 0:
#             limit = float(np.max(np.abs(values))) or 1.0
#         return -limit, limit
#     lower, upper = np.quantile(values, [0.01, 0.99])
#     if np.isclose(lower, upper):
#         lower, upper = float(np.min(values)), float(np.max(values))
#     if np.isclose(lower, upper):
#         upper = float(lower) + 1.0
#     return float(lower), float(upper)


# def draw_global_panel(
#     ax, data: xr.DataArray, *, title: str, cmap: str, unit: str,
#     vmin: float, vmax: float, panel_label: str, extend: str = 'neither',
# ):
#     mesh = ax.pcolormesh(
#         data['lon'], data['lat'], data,
#         transform=ccrs.PlateCarree(), shading='auto', rasterized=True,
#         cmap=cmap, vmin=vmin, vmax=vmax,
#     )
#     ax.set_global()
#     ax.set_facecolor('#F7FAFC')
#     ax.coastlines(resolution='110m', linewidth=0.55, color='#2F3437')
#     ax.gridlines(linewidth=0.35, color='#9CA3AF', alpha=0.45, linestyle=':')
#     ax.spines['geo'].set_edgecolor('#333333')
#     ax.spines['geo'].set_linewidth(0.85)
#     ax.set_title(title, pad=7, fontsize=11, fontweight='semibold')
#     ax.text(
#         0.015, 0.985, panel_label, transform=ax.transAxes,
#         va='top', ha='left', fontsize=10, fontweight='bold', color='#222222',
#         bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.78, 'pad': 2},
#     )
#     colorbar = ax.figure.colorbar(
#         mesh, ax=ax, orientation='horizontal', shrink=0.82, pad=0.035,
#         aspect=30, extend=extend,
#     )
#     colorbar.set_label(unit, fontsize=9, labelpad=2)
#     colorbar.outline.set_edgecolor('#555555')
#     colorbar.outline.set_linewidth(0.55)
#     colorbar.ax.tick_params(labelsize=8, length=2.5, width=0.5)
#     if not bool(np.isfinite(data).any()):
#         ax.text(
#             0.5, 0.5, 'Unavailable', transform=ax.transAxes,
#             ha='center', va='center', fontsize=11, color='#374151',
#             bbox={'facecolor': 'white', 'edgecolor': '#9CA3AF', 'alpha': 0.9},
#         )


# # ── Single per-model loop: load from disk → QC stats → summary → maps → release ──
# projection = ccrs.Robinson()
# qc_rows = []
# summary_rows = []

# for run in PROCESSED:
#     model = run['model']
#     processed_dir = run['run_root'] / 'processed'
#     qc_dir = run['run_root'] / 'qc'
#     qc_dir.mkdir(parents=True, exist_ok=True)
#     period = f'{START_YEAR}–{END_YEAR}'

#     climatology = xr.open_dataset(
#         processed_dir / f'global_climatology_{START_YEAR}_{END_YEAR}.nc'
#     ).load()
#     annual = xr.open_dataset(
#         processed_dir / f'global_annual_{START_YEAR}_{END_YEAR}.nc'
#     ).load()

#     # ── QC stats ──
#     model_qc_rows = []
#     for name in climatology.data_vars:
#         if name in FIXED_VAR_NAMES or name == 'WB':
#             continue
#         values = climatology[name]
#         model_qc_rows.append({
#             'model': model, 'variable': name,
#             'finite_cells': int(np.isfinite(values).sum()),
#             'missing_cells': int(values.isnull().sum()),
#             'minimum': float(values.min(skipna=True)),
#             'median': float(values.median(skipna=True)),
#             'maximum': float(values.max(skipna=True)),
#             'negative_cells': int((values < 0).sum()),
#         })
#     if 'WB' in climatology:
#         values = climatology['WB']
#         model_qc_rows.append({
#             'model': model, 'variable': 'WB',
#             'finite_cells': int(np.isfinite(values).sum()),
#             'missing_cells': int(values.isnull().sum()),
#             'minimum': float(values.min(skipna=True)),
#             'median': float(values.median(skipna=True)),
#             'maximum': float(values.max(skipna=True)),
#             'negative_cells': int((values < 0).sum()),
#         })
#     if 'ET' in annual:
#         print(
#             f"{model} negative ET: "
#             f"annual values={int((annual['ET'] < 0).sum())}, "
#             f"climatological cells={int((climatology['ET'] < 0).sum())}"
#         )
#     qc_rows.extend(model_qc_rows)
#     pd.DataFrame(model_qc_rows).to_csv(qc_dir / 'climatology_qc.csv', index=False)

#     del annual

#     # ── Summary row ──
#     ds = climatology
#     nlat = len(ds['lat'])
#     nlon = len(ds['lon'])
#     lat_vals = ds['lat'].values
#     lon_vals = ds['lon'].values
#     dlat = abs(float(np.median(np.diff(lat_vals)))) if nlat > 1 else float('nan')
#     dlon = abs(float(np.median(np.diff(lon_vals)))) if nlon > 1 else float('nan')
#     time_vars = sorted(v for v in ds.data_vars if v not in FIXED_VAR_NAMES and v != 'WB')
#     summary_rows.append({
#         'model': run['model'],
#         'family': run['family'],
#         'pool': run['pool'],
#         'member': run['member_id'],
#         'grid_label': run['grid_label'],
#         'nlat': nlat,
#         'nlon': nlon,
#         'dlat_deg': round(dlat, 3),
#         'dlon_deg': round(dlon, 3),
#         'resolution_approx': f'~{round((dlat + dlon) / 2, 1)}°',
#         'n_time_vars': len(time_vars),
#         'variables': ', '.join(time_vars),
#         'has_ET': 'ET' in ds,
#         'has_WB': 'WB' in ds,
#         'land_ice_status': ds.attrs.get('land_ice_status', 'unknown'),
#         'processed_status': run['processed_status'],
#         'output_dir': str(run['run_root']),
#     })

#     # ── Water-cycle maps ──
#     water_specs = []
#     if 'P' in ds:
#         water_specs.append(('P', 'Precipitation (P)', 'Blues', 'mm yr⁻¹', False))
#     if 'ET' in ds:
#         water_specs.append(('ET', 'Evapotranspiration (ET)', 'YlGn', 'mm yr⁻¹', False))
#     if 'R' in ds:
#         water_specs.append(('R', 'Runoff (R)', 'PuBu', 'mm yr⁻¹', False))
#     if 'WB' in ds:
#         water_specs.append(('WB', 'Water-balance residual (P−ET−R)', 'RdBu_r', 'mm yr⁻¹', True))

#     n_panels = len(water_specs)
#     if n_panels > 0:
#         fig, axes = plt.subplots(
#             1, n_panels, figsize=(3.5 * n_panels, 3),
#             subplot_kw={'projection': projection}, constrained_layout=True,
#         )
#         if n_panels == 1:
#             axes = [axes]
#         for panel, (ax, (name, title, cmap, unit, symmetric)) in enumerate(
#             zip(axes, water_specs)
#         ):
#             vmin, vmax = robust_limits(ds[name], symmetric=symmetric)
#             draw_global_panel(
#                 ax, ds[name], title=title, cmap=cmap, unit=unit,
#                 vmin=vmin, vmax=vmax, panel_label=chr(97 + panel), extend='both',
#             )
#         fig.suptitle(
#             f'Water-cycle diagnostics · {model} · {EXPERIMENT_ID} · {period}',
#             fontsize=13, fontweight='semibold', color='#262626',
#         )
#         fig.savefig(qc_dir / 'global_climatology_maps.png', dpi=180, bbox_inches='tight')
#         plt.show()

#     # ── Fixed-field maps ──
#     fixed_specs = [
#         (ds['cell_area'] / 1e10, 'Grid-cell area', 'cividis', '10¹⁰ m²', None),
#         (ds['land_fraction'], 'Land fraction (sftlf)', 'YlGn', '%', (0.0, 100.0)),
#         (ds['land_ice_fraction'], 'Land-ice fraction (sftgif)', 'Blues', '%', (0.0, 100.0)),
#     ]
#     fig, axes = plt.subplots(
#         1, 3, figsize=(14, 4), subplot_kw={'projection': projection}, constrained_layout=True,
#     )
#     for panel, (ax, (data, title, cmap, unit, limits)) in enumerate(
#         zip(np.atleast_1d(axes).ravel(), fixed_specs)
#     ):
#         vmin, vmax = limits if limits is not None else robust_limits(data)
#         draw_global_panel(
#             ax, data, title=title, cmap=cmap, unit=unit,
#             vmin=vmin, vmax=vmax, panel_label=chr(97 + panel),
#         )
#     fig.suptitle(
#         f'Fixed-grid diagnostics · {model} · {EXPERIMENT_ID}',
#         fontsize=13, fontweight='semibold', color='#262626',
#     )
#     fig.savefig(qc_dir / 'global_fixed_fields.png', dpi=180, bbox_inches='tight')
#     plt.show()

#     n_total = len([v for v in ds.data_vars if v not in FIXED_VAR_NAMES])
#     print(f'{model}: {n_total} variables in climatology')

#     del ds, climatology
#     plt.close('all')
#     gc.collect()

# # ── Aggregate tables ──
# QC = pd.DataFrame(qc_rows)
# display(QC)
# if not QC.empty:
#     QC = _merge_and_save(QC, DATA_ROOT / 'global_climatology_qc_summary.csv', ['model', 'variable'])
#     print(f'QC summary: {QC["model"].nunique()} models total (merged with previous runs)')

# SUMMARY = pd.DataFrame(summary_rows)
# if not SUMMARY.empty:
#     SUMMARY = _merge_and_save(SUMMARY, DATA_ROOT / 'S1_processed_summary.csv', ['model'])
#     print(f'Processed summary: {SUMMARY["model"].nunique()} models total')
# display(SUMMARY)

## 6. Handoff / 后续交接

SSP products are stored under `/Volumes/mimi-T9/CMIP6/{model}/{ssp scenario}/{member}/{grid}/`. Each model × scenario directory contains raw files, and (after processing is enabled) a global annual NetCDF, global climatology NetCDF, climatology Parquet, and QC records.

**Workflow:** Run this notebook once per scenario — set `CURRENT_SCENARIO` to `ssp126`, `ssp245`, or `ssp585` and execute all cells. Fixed fields are reused from historical downloads automatically. To process additional families, add them to `BATCH_FAMILIES` or names to `EXTRA_MODELS` and re-run.

**中文说明：** SSP产品保存在`/Volumes/mimi-T9/CMIP6/{model}/{ssp scenario}/...`目录下。每个情景运行一次本Notebook——修改`CURRENT_SCENARIO`后执行所有cell。固定场自动复用历史下载。要处理更多家族，将家族名加入`BATCH_FAMILIES`或模型名加入`EXTRA_MODELS`并重新运行。